# 모델 성능 개선

## 가. 모델 평가 방법의 개선
### 1. **지양해야 할 평가 방법** :
#### 가) 전체 데이터를 학습용(Train, 60~80%)과 평가용(Test, 20~40%)으로 단 한 번만 무작위 분할하여 평가하는 방식
### 2. **K-Fold 교차 검증 (Cross-Validation)** :
#### 가) **개념** : "운 좋게 맞춘 건가, 진짜 실력인가?"를 판별하는 표준 기법
#### 나) **작동 절차** :
1. Train 데이터를 $k$개의 그룹으로 나눈다.
2. $k-1$개의 그룹을 학습에 사용한다.
3. 나머지 1개의 그룹(Validation Set)을 이용해서 평가를 수행한다.
4. 2번, 3번 과정을 $k$번 반복한다.
5. 모든 결과의 평균을 구한다.

In [1]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np

# 1. 데이터 로드 및 KFold 객체 생성 (k=5, 섞기(shuffle) 적용)
iris = load_iris()
X, y = iris.data, iris.target

kfold = KFold(n_splits=5, shuffle=True, random_state=0)
cv_accuracy = []
fold_num = 1

# 2. 반복문을 통한 교차 검증 수행
for train_index, test_index in kfold.split(X):
    # 학습용, 검증용 데이터 분할
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # 모델 생성 및 학습
    dt = DecisionTreeClassifier(random_state=0)
    dt.fit(X_train, y_train)
    
    # 예측 및 평가
    pred = dt.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    cv_accuracy.append(accuracy)
    
    print(f"{fold_num}회 차 검증 정확도: {accuracy:.4f}")
    fold_num += 1

# 3. 최종 평균 정확도 계산
print("-" * 40)
print(f"평균 검증 정확도: {np.mean(cv_accuracy):.4f}")

1회 차 검증 정확도: 1.0000
2회 차 검증 정확도: 0.9000
3회 차 검증 정확도: 1.0000
4회 차 검증 정확도: 0.9000
5회 차 검증 정확도: 0.9333
----------------------------------------
평균 검증 정확도: 0.9467


In [2]:
# [사전 준비] 데이터 로드 및 학습/평가 데이터 분할 (분류용: 유방암 데이터)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# 2. 학습용(Train)과 평가용(Test) 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## 나. GridSearchCV를 이용한 모델 성능 고도화
### 1. **개념** : Scikit-learn에서 제공하는 모델 성능 극대화를 위한 최적의 하이퍼파라미터 탐색 및 교차 검증 도구
### 2. **특징** : 사용자가 제시한 하이퍼파라미터 탐색 범위 내의 **모든 조합**에 대해 학습과 평가를 수행하여 가장 성능이 좋은 최적의 하이퍼파라미터 값을 도출

In [3]:
# [실습] GridSearchCV를 이용한 분류 모델(Decision Tree) 성능 고도화
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# 1. 성능 고도화 대상 모델 생성
dt = DecisionTreeClassifier(random_state=0)

# 2. 최적화할 매개변수와 탐색 조건 설정 (Python Dictionary)
params = {
    'max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_leaf': [2, 3, 4, 5]
}

# 3. GridSearchCV 탐색 도구 생성
grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=params,
    scoring='accuracy',
    cv=5
)

# 4. 탐색 도구 학습 및 평가
grid_dt.fit(X_train, y_train)

# 5. Best 모델 추출 및 예측/평가
best_dt = grid_dt.best_estimator_
pred_test_dt = best_dt.predict(X_test)
acc_dt = accuracy_score(y_test, pred_test_dt)
print(f"GridSearchCV 최적 정확도: {acc_dt:.4f}")

GridSearchCV 최적 정확도: 0.9474


In [4]:
# [실습] GridSearchCV를 이용한 회귀 모델(Decision Tree Regressor) 성능 고도화
from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error as rmse

# 1. 회귀 데이터 로드 및 분할
housing = fetch_california_housing()
X_r, y_r = housing.data, housing.target
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=0)

# 2. 성능 고도화 대상 회귀 모델 생성
dt_reg = DecisionTreeRegressor(random_state=0)

# 3. 최적화할 매개변수 설정
params = {
    'max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_leaf': [2, 3, 4, 5]
}

# 4. GridSearchCV 탐색 도구 생성 (회귀 평가 지표 적용)
grid_dt_reg = GridSearchCV(
    estimator=dt_reg,
    param_grid=params,
    scoring='neg_root_mean_squared_error',
    cv=5
)

# 5. 학습 및 평가
grid_dt_reg.fit(X_train_r, y_train_r)
best_dt_reg = grid_dt_reg.best_estimator_
pred_test_reg = best_dt_reg.predict(X_test_r)
rmse_score = rmse(y_test_r, pred_test_reg)
print(f"GridSearchCV 최적 RMSE: {rmse_score:.4f}")

GridSearchCV 최적 RMSE: 0.6400


## 다. RandomizedSearchCV를 이용한 모델 성능 고도화
### 1. **개념** : Scikit-learn에서 제공하는 최적의 하이퍼파라미터 탐색 및 교차 검증 도구
### 2. **특징** : 사용자가 제시한 하이퍼파라미터 범위 내의 모든 조합을 탐색하는 대신, 설정한 횟수(`n_iter`)만큼 **무작위로 샘플링**하여 탐색을 수행하므로 탐색 시간을 대폭 단축

In [5]:
# [실습] RandomizedSearchCV를 이용한 분류 모델 성능 고도화
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

# 1. 모델 생성
dt = DecisionTreeClassifier(random_state=0)

# 2. 탐색할 매개변수 분포 설정
param_distributions = {
    'max_depth': range(3, 21),
    'min_samples_leaf': range(2, 11)
}

# 3. RandomizedSearchCV 탐색 도구 생성
random_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='accuracy',
    cv=5,
    random_state=0
)

# 4. 학습 및 평가
random_dt.fit(X_train, y_train)
best_dt_random = random_dt.best_estimator_
pred_test_random = best_dt_random.predict(X_test)
acc_random = accuracy_score(y_test, pred_test_random)
print(f"RandomizedSearchCV 최적 정확도: {acc_random:.4f}")

RandomizedSearchCV 최적 정확도: 0.9474


In [6]:
# [실습] RandomizedSearchCV를 이용한 회귀 모델 성능 고도화
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error as rmse

# 1. 회귀 모델 생성
dt_reg = DecisionTreeRegressor(random_state=0)

# 2. 매개변수 분포 설정
param_distributions = {
    'max_depth': range(3, 21),
    'min_samples_leaf': range(2, 11)
}

# 3. RandomizedSearchCV 탐색 도구 생성
random_dt_reg = RandomizedSearchCV(
    estimator=dt_reg,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=0
)

# 4. 학습 및 평가
random_dt_reg.fit(X_train_r, y_train_r)
best_reg_random = random_dt_reg.best_estimator_
pred_reg_random = best_reg_random.predict(X_test_r)
rmse_random = rmse(y_test_r, pred_reg_random)
print(f"RandomizedSearchCV 최적 RMSE: {rmse_random:.4f}")

RandomizedSearchCV 최적 RMSE: 0.6055


In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. 데이터 로드 및 학습/평가용 분할 (분류 예시: 유방암 데이터)
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=0, stratify=cancer.target
)

# 2. 성능 고도화 대상 모델 생성
rf = RandomForestClassifier(random_state=0)

# 3. 최적화할 하이퍼파라미터와 탐색 범위 설정 (Dictionary 형태)
params = {
    'n_estimators': [50, 100, 200],         # 숲을 구성하는 결정 트리의 개수
    'max_depth': [None, 5, 10, 15],          # 트리의 최대 깊이 (과대적합 방지)
    'min_samples_split': [2, 5, 10],         # 노드를 분할하기 위한 최소 샘플 수
    'min_samples_leaf': [1, 2, 4],           # 리프 노드가 가져야 할 최소 샘플 수
    'max_features': ['sqrt', 'log2']         # 분할 시 고려할 최대 피터 개수
}

# 4. GridSearchCV 탐색 도구 생성 (5-Fold 교차 검증 수행)
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=params,
    scoring='accuracy',  # 평가 지표
    cv=5,                # 교차 검증 폴드 수
    n_jobs=-1            # 모든 CPU 코어 활용하여 빠른 탐색
)

# 5. 탐색 도구 학습 (모든 조합을 돌며 최적의 파라미터 탐색)
print("랜덤 포레스트 하이퍼파라미터 최적화(GridSearchCV) 진행 중...")
grid_rf.fit(X_train, y_train)

# 6. 최적의(Best) 모델 추출 및 파라미터 확인
best_rf = grid_rf.best_estimator_
print("\n[탐색 완료]")
print(f"최적의 하이퍼파라미터: {grid_rf.best_params_}")
print(f"교차 검증 최고 정확도: {grid_rf.best_score_:.4f}")

# 7. 평가용(Test) 데이터를 이용한 최종 예측 및 성능 평가
pred_test = best_rf.predict(X_test)
test_accuracy = accuracy_score(y_test, pred_test)

print(f"평가용 데이터 최종 정확도: {test_accuracy:.4f}")
print("\n[상세 분류 보고서]")
print(classification_report(y_test, pred_test))

랜덤 포레스트 하이퍼파라미터 최적화(GridSearchCV) 진행 중...

[탐색 완료]
최적의 하이퍼파라미터: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
교차 검증 최고 정확도: 0.9758
평가용 데이터 최종 정확도: 0.9561

[상세 분류 보고서]
              precision    recall  f1-score   support

           0       0.93      0.95      0.94        42
           1       0.97      0.96      0.97        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

